# Implicit modelling with a deep GP

In [ ]:
%%capture
!pip install cmcrameri # scientific color maps
!pip install git+https://github.com/italo-goncalves/geoML.git@claude

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cmcrameri import cm
import pyvista as pv
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import geoml

import geoml.kernels as kr
import geoml.transform as tr
import geoml.latent as gl
import geoml.likelihood as lk

## The dataset

This dataset contains a set of drillholes intersecting a quartz vein. The objective is to model a geometry of this vein using a variety of methods, inlcuding a deep GP.

In [ ]:
collar = pd.read_excel('https://drive.google.com/uc?export=download&id=1AT-Hp540CzBP6UFDt6X7MwNTgQ0NA8Gs')
collar.head(10)

In [ ]:
lito = pd.read_excel('https://drive.google.com/uc?export=download&id=1ktNiQmvsETtIlXu1fhHFW3jzHRFI0BvW')
lito.head(10)

geoML supports vertical drillhole databases. The two tabels are merged to obtain the data points in 3D.

In [ ]:
dh_raw = geoml.drillhole.DrillholeData(collar, hole="HOLE", length='TDEPTH',
                                       x="EAST", y="NORTH", z="RL")
dh_raw.add_intervals("SIMPLE LITO", lito, hole="HOLE")
dh_raw

In [ ]:
# Merging redundant segments
merged = dh_raw.merge_domains(("SIMPLE LITO", "SIMPLE LITO"))
dh_clean = dh_raw.add_intervals("SIMPLE LITO", merged)

# A sparse set of points to be used as inducing points
dh_sparse = dh_clean.as_classification_input(("SIMPLE LITO", "SIMPLE LITO"), length=10)

# A dense set for training
dh_point = dh_clean.as_classification_input(("SIMPLE LITO", "SIMPLE LITO"), length=1)

dh_point

In [ ]:
print(dh_point.tree())

In [ ]:
import plotly.graph_objects as go

vein_color = {"Vein": "yellow", "Waste": "gray"}

# Extracting coordinates and labels directly from the PointData object
coords = dh_point.coordinates
unique_labels = np.array(dh_point.get('SIMPLE LITO/measurements_a').labels)
lito_labels = unique_labels[dh_point.get('SIMPLE LITO/measurements_a').values.to_numpy()]


fig = go.Figure()

for label in unique_labels:
    # Create a mask for each lithology type
    mask = lito_labels == label

    fig.add_trace(go.Scatter3d(
        x=coords[mask, 0],
        y=coords[mask, 1],
        z=coords[mask, 2],
        mode='markers',
        marker=dict(size=2, color=vein_color[label]),
        name=label
    ))

fig.update_layout(
    scene=dict(
        xaxis_title='East',
        yaxis_title='North',
        zaxis_title='RL',
        aspectmode='data'
    ),
    width=900,
    height=700
)

fig.show()

## A stationary model

A simple, stationary VGP model to serve as a baseline. The anisotropy ellipsoid was determined visually and initialized with the parameters below. They will be able to change during training.

In [ ]:
stat_input = gl.BasicInput(
    inducing_points=dh_sparse,
    transform=tr.Anisotropy3D(100, 1.0, 0.5, azimuth=270, dip=45),
    center=True)
stat_gp = gl.BasicGP(stat_input, size=1)
stat_output = gl.Linear(stat_gp, size=2)

stat_model = geoml.models.VGPNetwork(
    data=dh_point,
    variables="SIMPLE LITO",
    likelihoods=geoml.likelihood.CategoricalGaussianIndicator(2),
    latent_network=stat_output,
    options=geoml.models.GPOptions(jitter=1e-6, training_batch_size=500))

# stat_model.set_learning_rate(5e-2)
# stat_model.train_svi(20)
stat_model.set_learning_rate(2e-2)
# stat_model.train_svi(50)
stat_model.train_full(1000)

In [ ]:
plt.figure(figsize=[10, 6])
plt.plot(stat_model.training_log)
plt.xlabel('Iteration')
plt.ylabel('ELBO')
plt.show()

In [ ]:
stat_grid = geoml.data.Grid3D(start=[24850, 15700, 1300],
                          end=[25150, 16050, 1600],
                          n=[121, 141, 121])
stat_model.predict(stat_grid, n_sim=1)

After predicting on the grid, an isosurface is extracted and the model is used again to transfer the properties to the isosurface's points.

In [ ]:
stat_surf = stat_grid.variables['SIMPLE LITO'].components['Vein'].indicator_predicted.get_contour(0.001)
stat_model.predict(stat_surf)
pv_stat_surf = stat_surf.as_pyvista()

In [ ]:
fig = go.Figure()

for label in unique_labels:
    mask = lito_labels == label
    fig.add_trace(go.Scatter3d(
        x=coords[mask, 0], y=coords[mask, 1], z=coords[mask, 2],
        mode='markers',
        marker=dict(size=2, color=vein_color[label]),
        name=f"DH: {label}"
    ))

# Extract vertices and faces from the pyvista object
vertices = pv_stat_surf.points
# PyVista faces are [n_points, p1, p2, p3, ...], for triangles it's [3, p1, p2, p3]
faces = pv_stat_surf.faces.reshape(-1, 4)[:, 1:]
uncertainty = pv_stat_surf.point_data['SIMPLE LITO - uncertainty']

fig.add_trace(go.Mesh3d(
    x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    intensity=uncertainty,
    colorscale='Turbo',
    opacity=1.0,
    name='Vein Surface (Uncertainty)',
    showscale=True,
    colorbar=dict(title="Uncertainty")
))

fig.update_layout(
    scene=dict(
        xaxis_title='East', yaxis_title='North', zaxis_title='RL',
        aspectmode='data'
    ),
    width=1000, height=800,
    margin=dict(l=0, r=0, b=0, t=40),
    title="Drillholes and Vein Surface (Stationary Model)"
)

fig.show()

A confusion matrix can be drawn to check model performance.

In [ ]:
stat_model.predict(dh_point)
keep = ~dh_point.variables["SIMPLE LITO"].boundary.values.to_numpy().astype(bool)
conf = confusion_matrix(
    dh_point.variables["SIMPLE LITO"].measurements_a.values[keep],
    dh_point.variables["SIMPLE LITO"].predicted.values[keep])
conf_disp = ConfusionMatrixDisplay(
    conf, display_labels=dh_point.variables["SIMPLE LITO"].labels[::-1])
conf_disp.plot()

## Deep spatial learning

Now we build a deep learning model. It apply a stochastic differential equation (SDE) to make the points move in space, resulting in non-stationarity. This model will be trained using Stochastic Variational Inference (SVI) to work with minibatches.

In [ ]:
deep_input = gl.BasicInput(
    inducing_points=dh_sparse,
    transform=tr.Anisotropy3D(100, 1.0, 0.5, azimuth=270, dip=45),
    center=True)

# SDE requires a vector field in 3D to induce movement to the points
field = gl.BasicGP(deep_input, size=3)

# The GPWalk node uses the field to move the points
new_coords = gl.GPWalk(field, n_steps=5)

# Now the implicit model is based on the transformed coordinates
deep_gp = gl.BasicGP(new_coords, size=1)
deep_output = gl.Linear(deep_gp, size=2)

deep_model = geoml.models.VGPNetwork(
    data=dh_point,
    variables="SIMPLE LITO",
    likelihoods=geoml.likelihood.CategoricalGaussianIndicator(2),
    latent_network=deep_output,
    options=geoml.models.GPOptions(jitter=1e-6, training_batch_size=500))


# Deep models may have more local optima,
# so the optimizer is reset after some epochs
deep_model.set_learning_rate(5e-2)
deep_model.train_svi(20)
deep_model.set_learning_rate(1e-2)
deep_model.train_svi(60)

In [ ]:
plt.figure(figsize=[10, 6])
plt.plot(stat_model.training_log, label='Stationary')
plt.plot(deep_model.training_log, label='Non-stationary')
plt.legend()
plt.xlabel('Iteration')
plt.ylabel('ELBO')
plt.show()

In [ ]:
deep_grid = geoml.data.Grid3D(start=[24850, 15700, 1300],
                          end=[25150, 16050, 1600],
                          n=[121, 141, 121])
deep_model.predict(deep_grid, n_sim=1)

In [ ]:
deep_surf = deep_grid.variables['SIMPLE LITO'].components['Vein'].indicator_predicted.get_contour(0.001)
deep_model.predict(deep_surf)
pv_deep_surf = deep_surf.as_pyvista()

In [ ]:
fig = go.Figure()

for label in unique_labels:
    mask = lito_labels == label
    fig.add_trace(go.Scatter3d(
        x=coords[mask, 0], y=coords[mask, 1], z=coords[mask, 2],
        mode='markers',
        marker=dict(size=2, color=vein_color[label]),
        name=f"DH: {label}"
    ))

# Extract vertices and faces from the pyvista object
vertices = pv_deep_surf.points
# PyVista faces are [n_points, p1, p2, p3, ...], for triangles it's [3, p1, p2, p3]
faces = pv_deep_surf.faces.reshape(-1, 4)[:, 1:]
uncertainty = pv_deep_surf.point_data['SIMPLE LITO - uncertainty']

fig.add_trace(go.Mesh3d(
    x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    intensity=uncertainty,
    colorscale='Turbo',
    opacity=1.0,
    name='Deep Vein Surface',
    showscale=True,
    colorbar=dict(title="Uncertainty")
))

fig.update_layout(
    scene=dict(
        xaxis_title='East', yaxis_title='North', zaxis_title='RL',
        aspectmode='data'
    ),
    width=1000, height=800,
    margin=dict(l=0, r=0, b=0, t=40),
    title="Drillholes and Vein Surface (Deep Model)"
)

fig.show()

In [ ]:
deep_model.predict(dh_point)
keep = ~dh_point.variables["SIMPLE LITO"].boundary.values.to_numpy().astype(bool)
conf = confusion_matrix(
    dh_point.variables["SIMPLE LITO"].measurements_a.values[keep],
    dh_point.variables["SIMPLE LITO"].predicted.values[keep])
conf_disp = ConfusionMatrixDisplay(
    conf, display_labels=dh_point.variables["SIMPLE LITO"].labels[::-1])
conf_disp.plot()